# Lab 25-04 · Dies, Defects and Dollars

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Circle, Rectangle
from scipy import optimize
import yieldcost as y

## Part 1 · Counting dies

*Predict* whether an optimized grid fits more than the formula's 601 dies.

In [2]:
n, ox, oy = y.dpw_exact(294, 10.1, 10.1)
print(f"formula {y.dpw_formula(294, 102.01):.1f}; exact best {n} at offset ({ox:.2f}, {oy:.2f}) mm")
sites = y.die_sites(294, 10.1, 10.1, ox, oy)
fig, ax = plt.subplots(figsize=(5, 5))
ax.add_patch(Circle((0, 0), 147, fill=False))
for sx, sy in sites:
    ax.add_patch(Rectangle((sx, sy), 10.1, 10.1, fill=False, lw=0.3))
ax.set_xlim(-155, 155); ax.set_ylim(-155, 155); ax.set_aspect("equal")
plt.show()

formula 600.8; exact best 612 at offset (0.51, 5.55) mm


## Part 2 · Clustering

*Predict* the uniform and clustered yields at $AD_0$ = 1.

In [3]:
for alpha in (None, 3, 1):
    sim = y.simulate_yield(1.0, 1.0, alpha)
    ref = y.y_poisson(1.0) if alpha is None else y.y_negbin(1.0, alpha)
    print(f"alpha={alpha}: simulated {sim:.4f}, formula {ref:.4f}")

alpha=None: simulated 0.3678, formula 0.3679
alpha=3: simulated 0.4219, formula 0.4219
alpha=1: simulated 0.5007, formula 0.5000


## Part 3 · Cost against area

*Predict* the area at which cost per good mm² doubles relative to 50 mm².

In [4]:
base = y.cost_per_good_mm2(20000, 300, 50, 0.1, 3)
a2 = optimize.brentq(lambda a: y.cost_per_good_mm2(20000, 300, a, 0.1, 3) - 2 * base, 60, 800)
print(f"50 mm2: ${base:.3f} per good mm2; doubles at {a2:.0f} mm2")
areas = np.linspace(20, 850, 200)
plt.figure()
plt.plot(areas, [y.cost_per_good_mm2(20000, 300, a, 0.1, 3) for a in areas])
plt.xlabel("die area (mm2)"); plt.ylabel("$ per good mm2"); plt.grid(True)
plt.show()

50 mm2: $0.319 per good mm2; doubles at 602 mm2


## Part 4 · Learning

*Predict* the 80% month with a 3-month time constant.

In [5]:
for tau in (6, 3):
    print(f"tau = {tau} months: 80% yield at month {y.learning_month(0.8, 1.0, 3, 0.08, 0.5, tau):.2f}")
t = np.linspace(0, 24, 200)
plt.figure()
for tau in (6, 3):
    d = 0.08 + 0.42 * np.exp(-t / tau)
    plt.plot(t, [y.y_negbin(di, 3) for di in d], label=f"tau = {tau} months")
plt.axhline(0.8, color="0.5", ls=":")
plt.xlabel("months"); plt.ylabel("yield, 1 cm2 die"); plt.legend(); plt.grid(True)
plt.show()

tau = 6 months: 80% yield at month 6.11
tau = 3 months: 80% yield at month 3.06
